# Subtitles Showcase

This notebook demonstrates how to extract and process text from `.srt` subtitle files, clean the text, remove duplicates, and finally extract vocabulary tokens (verbs and other words) using the Natural Language API.

In [ ]:
%load_ext autoreload

: 

In [ ]:
%autoreload 2
from setup_imports import *  # noqa: F401,F403


from src.subtitles import read_subtitles, process_subtitles, get_subtitle_tokens
from src.nlp import get_text_tokens, get_verbs_and_vocab
from src.phrases.phrase_model import get_unique_tokens_from_phrases
from src.phrases.search import get_phrases_by_collection, find_phrases_by_vocab_dict

: 

## 1. Reading Subtitles
We use `pysrt` to read the subtitles from the `data/subtitles/` directory.

In [ ]:
subtitle_file = "../data/subtitles/Trouble_sv_only.srt"

try:
    subs = read_subtitles(subtitle_file)
    print(f"Successfully loaded {len(subs)} subtitle entries.")
    print("\nFirst 3 entries:")
    for i in range(min(3, len(subs))):
        print(f"{i+1}: {subs[i].text}")
except Exception as e:
    print(f"Error reading subtitles: {e}")

: 

## 2. Processing Subtitles
We will now clean the subtitles by removing Closed Captioning (CC) info enclosed in square brackets `[like this]`, replace newlines with spaces, strip extra whitespace, and remove duplicates to get a unique list of phrases.

In [18]:
unique_phrases = process_subtitles(subs)

print(f"Total unique phrases extracted: {len(unique_phrases)}")
print("\nSample of 10 unique phrases:")
for phrase in unique_phrases[:10]:
    print(f"- {phrase}")

Total unique phrases extracted: 1315

Sample of 10 unique phrases:
- Men snälla, tyst
- Du väntar här. När jag messar dig kommer du in med resten av grejerna
- Ja. Capite
- Ska vi sticka
- Vad är det här för nåt
- Det är ett kilo, det här. Vi sa 30. - Mm
- Vi har dem. Men pengarna först
- Vi står utanför Bamboo Bamboo
- Det är bekräftat: de misstänkta befinner sig i lokalen
- Kör


## 3. Extracting Tokens
Using `src.nlp`, we'll run a sample of these phrases through the Google Cloud Natural Language API to extract lemmas and group them into `verbs` and `vocab`.

In [19]:
# To avoid long API wait times in this showcase, we'll process just the first 20 phrases.
sample_phrases = unique_phrases[:20]

print(f"Extracting vocabulary from {len(sample_phrases)} phrases...")
vocab_dict = get_verbs_and_vocab(sample_phrases, language="sv-se")

print("\n--- Extracted Verbs ---")
print(vocab_dict.get('verbs', []))

print("\n--- Extracted Vocab (Non-Verbs) ---")
print(vocab_dict.get('vocab', []))

Extracting vocabulary from 20 phrases...

--- Extracted Verbs ---
['messa', 'skola', 'sticka', 'säga', 'plocka', 'ta', 'vänta', 'befinna', 'hända', 'komma', 'köra', 'kalla', 'ha', 'stå', 'vara', 'kunna']

--- Extracted Vocab (Non-Verbs) ---
['stopet', 'i', 'capite', 'för', 'på', 'de', 'först', 'men', 'när', '1490', 'aw', 'här', 'rest', 'med', 'en', 'uppfattat', '30', 'pengar', 'in', 'nåt', 'ni', 'utanför', 'ja', 'a', 'misstänkt', 'vi', 'lokal', 'sen', 'bamboo', 'då', 'fan', 'du', 'vad', 'portkoden', 'och', 'vänster', 'b', 'sig', 'kilo', 'mm', 'snäll', 'jag', 'grej', 'enhet', 'vapen', 'tyst', 'grabeb', 'så', 'bekräfta', 'av', 'den', 'bakväg']


In [29]:
p, m = find_phrases_by_vocab_dict(vocab_dict, language="sv-SE",collection="LM1000")

ValueError: too many values to unpack (expected 2)

In [26]:
[phrase.translations["sv-SE"].text for phrase in p]

['Det är snälla av dig att tänka så',
 'Vi ska ta itu med det',
 'Kommer de att tro på mig?',
 'Företaget har sitt säte i London',
 'Vad händer sen?',
 'Kan du växla pengar?',
 'Jag svänger till vänster här',
 'Ska vi vänta utanför?',
 'En pojke och en flicka',
 'Ett kort för varje person',
 'Ett lokalt produktionscenter',
 'Han kör en enkel bil',
 'Han står vid dörren',
 'Men datumet ändrades',
 'Jag smörjer in mig med solkräm',
 'Jag plockar äpplen till min lärare',
 'Jag ska säga något.',
 'Det var en misstänkt död',
 'Ingenting kommer att spela någon roll då.',
 'Hon sa ja igår.',
 'Det är en stor grej',
 'Det kalla vintervädret',
 'När började det?',
 'Kan ni ursäkta oss?',
 'Ska du knacka först?']

In [ ]:
film_tokens = get_subtitle_tokens(unique_phrases, "sv")

In [ ]:
sorted(film_tokens)

In [ ]:
lm1000 = get_phrases_by_collection("LM1000")

In [ ]:
lm1000_str = [(x._get_translation('sv-SE').text) for x in lm1000[:10]]

In [13]:
results = find_phrases_by_vocab_dict(
vocab_dict={"verbs": ["springa", "äta"], "vocab": ["hund", "bil"]},
language="sv-SE",
collection="LM1000")
for p in results:
    print(p.english, p.translations["sv-SE"].text)

2026-04-19 15:12:14 - audio-language-trainer - WARNING - nlp.py:151 - analyze_text_syntax: language not supported by Google NLP: sv-SE
2026-04-19 15:12:14 - audio-language-trainer - INFO - nlp.py:189 - extract_lemmas_and_pos: falling back to spaCy for language 'sv-SE'
2026-04-19 15:12:14 - audio-language-trainer - WARNING - nlp.py:197 - extract_lemmas_and_pos: neither Google NLP nor spaCy could process phrase for language 'sv-SE': Ett bättre alternativ
2026-04-19 15:12:14 - audio-language-trainer - WARNING - nlp.py:151 - analyze_text_syntax: language not supported by Google NLP: sv-SE
2026-04-19 15:12:14 - audio-language-trainer - INFO - nlp.py:189 - extract_lemmas_and_pos: falling back to spaCy for language 'sv-SE'
2026-04-19 15:12:14 - audio-language-trainer - WARNING - nlp.py:197 - extract_lemmas_and_pos: neither Google NLP nor spaCy could process phrase for language 'sv-SE': En cykel i låg växel
2026-04-19 15:12:14 - audio-language-trainer - WARNING - nlp.py:151 - analyze_text_synt

{'verbs': [],
 'vocab': [],
 'tokens': ['och',
  'rent',
  'flicka',
  'person',
  'under',
  'för',
  'kort',
  'billig',
  'bättre',
  'färg',
  'pappersark',
  'ljus',
  'förändring',
  'Ett',
  'i',
  'cykel',
  'bänken',
  'genomtänkt',
  'flaska',
  'pojke',
  'varje',
  'glasring',
  'En',
  'växel',
  'svar',
  'alternativ',
  'låg',
  'en',
  'väster']}

In [ ]:
get_

In [ ]:
lm1000_tokens = get_unique_tokens_from_phrases(lm1000, "sv-SE")
lm1000_tokens = list(map(lambda x: x.lower(), lm1000_tokens))

In [ ]:
len(film_tokens)

In [ ]:
len(lm1000_tokens)

In [ ]:
(set(map(lambda x: x[:-1],film_tokens)).difference(set(lm1000_tokens)))

In [ ]:
film_tokens